# MVO4 Swan 9: REPLICATION_GUIDE.md implementation

This revised notebook uses the shared `mvo_algo.py` pipeline. Run from the `replication` directory. The uploaded notebook is preserved separately. The guide controls all dates, swans, and model choices. No live downloads are used.

In [1]:
import os, sys
from pathlib import Path
for name in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[name] = "1"
HERE = Path.cwd()
if not (HERE / "mvo_algo.py").exists():
    raise RuntimeError("Run this notebook from the replication directory")
sys.path.insert(0, str(HERE))
import pandas as pd
import matplotlib.pyplot as plt
from mvo_algo import Replication, EVENTS, versions, write_outputs
model = Replication(cache=HERE / ".cache/dbi")
versions()

{'python': '3.12.14',
 'numpy': '2.2.3',
 'pandas': '2.3.3',
 'scikit-learn': '1.7.2',
 'scipy': '1.16.3',
 'PyPortfolioOpt': '1.6.0',
 'cvxpy': '1.9.2'}

## 1–3. Historical inputs, plain-average crash shape, and inclusive window

In [2]:
EVENT = 'Swan 9'
K, Seeds, MinAppear, w1 = 7, 3, 3, 0.17
ev = EVENTS[EVENT]
print(ev)
print("Eligible universe:", len(model.training_beta(EVENT)))
print("t1, t2, dd:", model.curve_params(EVENT))
print("start, end, extended:", model.training_window(EVENT))

Event(name='Swan 9', peak='2020-02-19', bottom='2020-03-23', train_swans=('Swan4', 'Swan5', 'Swan6', 'Swan7', 'Swan8'), optdata='revised_swan9_sp100_optdata4.xlsx')
Eligible universe: 86
t1, t2, dd: (72, 200, 0.24709028449533582)
start, end, extended: (Timestamp('2018-12-30 00:00:00'), Timestamp('2020-02-14 00:00:00'), False)


## 4. Rank all 1,000 seeds, vote, retain the top 22 with at least MinAppear votes

In [3]:
ranking = model.dbi_ranking(EVENT, K)
print("Best seeds:", ranking[:Seeds])
selected, votes = model.preselect(EVENT, K, Seeds, MinAppear)
display(votes.to_frame())
display(selected)
assert len(selected) >= 5, "This combination has no_preselect status"

Best seeds: [(335, 1.0620343324117123), (299, 1.0620343324117125), (969, 1.0620343324117125)]


,votes
ABT,3
AMGN,3
AMZN,3
CL,3
COST,3
EXC,3
JNJ,3
KO,3
LLY,3
MDLZ,3


,Swan4,Swan5,Swan6,Swan7,Swan8,avg
Ticker,,,,,,
ABT,0.702188,0.474696,0.311079,1.073150,0.431921,0.598607
AMGN,0.795355,0.502892,0.109292,0.843475,0.659977,0.582198
AMZN,-0.445606,0.328167,0.123350,0.334033,1.561997,0.380388
CL,0.099788,0.617542,-0.045657,0.784583,0.569564,0.405164
COST,0.829818,0.976294,-0.015790,0.618966,0.918643,0.665586
EXC,0.519667,0.794675,0.263389,0.605246,-0.123438,0.411908
JNJ,0.403161,0.739963,0.360346,0.668742,0.555245,0.545491
KO,-0.170562,0.536675,0.325968,0.614677,-0.049526,0.251446
LLY,0.746268,0.920577,0.280671,0.780342,0.061466,0.557865


## 5–6. Inject at the front, then compute last-year drawdown and weekly covariance ×51

In [4]:
risk = model.risk_inputs(EVENT, selected)
print("Trading days:", risk["total_days"], "Cutoff:", risk["cutoff"])
pd.Series(risk["curve"], index=risk["raw_prices"].index).plot(title="Injected market drawdown")
plt.show()
display(pd.DataFrame({"fall": risk["fall"], "R_fwd_30_MA": risk["R_fwd"], "E_2": risk["E_2"]}))
display(risk["cov"])

Trading days: 284 Cutoff: 2019-02-14 00:00:00


,fall,R_fwd_30_MA,E_2
Ticker,,,
ABT,0.100767,0.066674,0.661670
AMGN,0.107951,0.055532,0.514419
AMZN,0.146010,0.234882,1.608667
CL,0.123471,0.026415,0.213940
COST,0.060706,0.057972,0.954952
EXC,0.116306,0.066976,0.575866
JNJ,0.103603,0.049573,0.478491
KO,0.068427,-0.053464,-0.781324
LLY,0.124823,0.041847,0.335250


Ticker,ABT,AMGN,AMZN,CL,COST,EXC,JNJ,KO,LLY,MDLZ,MO,MRK,NEE,PEP,PFE,PG,SBUX,SO,VZ,WMT
Ticker,,,,,,,,,,,,,,,,,,,,
ABT,0.033672,0.021106,0.010156,0.008488,0.013875,0.008783,0.012406,0.009791,0.011253,0.012035,0.006097,0.020323,0.005118,0.007692,0.021510,0.007141,0.018136,0.003722,0.009552,0.003170
AMGN,0.021106,0.057776,0.003282,0.005845,0.006226,0.007019,0.013536,0.004110,0.023170,0.008067,-0.004305,0.029646,-0.000292,0.002753,0.013883,0.008343,0.011029,0.001337,0.007866,0.002064
AMZN,0.010156,0.003282,0.036795,0.012467,0.010140,0.008232,0.007329,0.007682,0.001792,0.010243,-0.003063,0.001702,-0.001018,0.007466,0.013151,0.005365,0.003990,-0.002092,0.006026,0.007101
CL,0.008488,0.005845,0.012467,0.026175,0.012737,0.014531,0.010397,0.012551,0.006342,0.016369,0.004092,0.005474,0.006370,0.012351,0.007489,0.014272,0.014534,0.002060,0.006582,0.008408
COST,0.013875,0.006226,0.010140,0.012737,0.029087,0.011937,0.009145,0.008048,0.001712,0.014119,0.005567,0.002521,0.001779,0.015407,0.011708,0.008333,0.015458,0.000672,0.011696,0.012077
EXC,0.008783,0.007019,0.008232,0.014531,0.011937,0.023905,0.012939,0.006165,0.010843,0.013843,0.002311,0.007222,0.008512,0.011984,0.009517,0.013416,0.006033,0.008556,0.010421,0.007307
JNJ,0.012406,0.013536,0.007329,0.010397,0.009145,0.012939,0.026303,0.007589,0.013038,0.012324,0.006146,0.015175,0.006415,0.013511,0.011703,0.009983,0.007164,0.006178,0.011668,0.007754
KO,0.009791,0.004110,0.007682,0.012551,0.008048,0.006165,0.007589,0.024946,0.004605,0.010792,0.003358,0.005390,0.008092,0.012086,0.012142,0.007797,0.016185,0.004980,0.008788,0.002098
LLY,0.011253,0.023170,0.001792,0.006342,0.001712,0.010843,0.013038,0.004605,0.041367,0.007917,-0.004589,0.019185,0.004936,0.002416,0.011796,0.000642,0.005278,0.006028,0.003125,-0.005723


## 7–8. Max Sharpe at rf=0, 3% floor, then buy at peak and evaluate at bottom

In [5]:
result = model.run_one(EVENT, K, Seeds, MinAppear, w1, detail=True)
print("Status:", result["status"], "PR:", result["PR"])
assert result["status"] == "ok", result.get("error")
display(result["weights"].to_frame("weight"))
pd.DataFrame({"portfolio": result["evaluation"]["portfolio_values"], "benchmark": result["evaluation"]["benchmark_values"]}).plot(title=EVENT)
plt.show()

Status: ok PR: 0.49119493879261333


,weight
AMZN,0.173111
COST,0.173111
LLY,0.070670
MDLZ,0.173111
MO,0.063776
MRK,0.173111
WMT,0.173111


## 9. Small sweep and optional exports

The command-line runner handles the complete 5,376-run grid. Below is a four-combination event sweep; failed optimization is retained as a status, with no alternate model.

In [6]:
small = model.run_sweep(dict(K=[7], Seeds=[3], MinAppear=[3], w1=[0.05, 0.11, 0.15, 0.17]), [EVENT])
display(small)
SAVE_OUTPUTS = False
if SAVE_OUTPUTS:
    output = HERE / "results" / EVENT.replace(" ", "_")
    write_outputs(small, output, [EVENT])
    model.export_checkpoints(output / "replication_checkpoints.xlsx", K=K, Seeds=Seeds, MinAppear=MinAppear, w1=w1)

Swan 9, K=7: 4 runs completed


,event,K,Seeds,MinAppear,w1,status,PR,port_dd,market_dd,n_preselect,n_final,holdings,error,train_start,train_end
0,Swan 9,7,3,3,0.05,ok,0.235135,0.241171,0.315312,20,20,"{""ABT"": 0.04999999999999999, ""AMGN"": 0.0499999...",None,2018-12-30,2020-02-14
1,Swan 9,7,3,3,0.11,ok,0.331725,0.210715,0.315312,20,9,"{""ABT"": 0.11520375354775196, ""AMZN"": 0.1152037...",None,2018-12-30,2020-02-14
2,Swan 9,7,3,3,0.15,ok,0.419273,0.183110,0.315312,20,8,"{""AMZN"": 0.15135767837502398, ""COST"": 0.151357...",None,2018-12-30,2020-02-14
3,Swan 9,7,3,3,0.17,ok,0.491195,0.160432,0.315312,20,7,"{""AMZN"": 0.17311080109568955, ""COST"": 0.173110...",None,2018-12-30,2020-02-14


Limitations: original checkpoint workbook is missing; fresh Yahoo downloads and alternative experimental objectives are outside this replication workflow. Swan9 analyst inputs have the guide's point-in-time caveat. See `IMPLEMENTATION_REPORT.md` for validation results.